**Install and load libraries**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install the evaluate library
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 579.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.8 MB/s eta 0:00:00


In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSequenceClassification

In [ ]:
# Load dataset
ds = load_dataset('imdb')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# Tokenize the data
def tokenize_data(examples):
    return tokenizer(examples['text'], padding = True, truncation = True)

tokenized_ds = ds.map(tokenize_data, batched = True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased-finetuned-sst-2-english')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
# Setup evaluation
metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

**Train the model**

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='model_dir',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    push_to_hub=False,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss
500,0.221000
1000,0.224900
1500,0.200900


TrainOutput(global_step=1563, training_loss=0.21521112816652577, metrics={'train_runtime': 1149.5866, 'train_samples_per_second': 21.747, 'train_steps_per_second': 1.36, 'total_flos': 3311684966400000.0, 'train_loss': 0.21521112816652577, 'epoch': 1.0})

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.18609359860420227, 'eval_f1': 0.933605311575074, 'eval_runtime': 411.3963, 'eval_samples_per_second': 60.769, 'eval_steps_per_second': 3.799, 'epoch': 1.0}


**Evaluate the model**

**Results analysis**

The model shows excellent performance.

- High F1 score of 0.9336 (close to 1) indicates that the model is effective in making accurate predictions across both positive and negative classes.

- A low evaluation loss of 0.1861 (close to the training loss of 0.2152) suggests the model is generalizing well to new data without much overfitting.

**Save the model**

In [ ]:
# Setting directory in Google Drive to save the model
model_dir = "/content/drive/MyDrive/Colab-Notebooks/LHL-LLM/fine_tuned_model"

# Save model and tokenizer
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

('/content/drive/MyDrive/Colab-Notebooks/LHL-LLM/fine_tuned_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab-Notebooks/LHL-LLM/fine_tuned_model/special_tokens_map.json',
 '/content/drive/MyDrive/Colab-Notebooks/LHL-LLM/fine_tuned_model/vocab.txt',
 '/content/drive/MyDrive/Colab-Notebooks/LHL-LLM/fine_tuned_model/added_tokens.json',
 '/content/drive/MyDrive/Colab-Notebooks/LHL-LLM/fine_tuned_model/tokenizer.json')